# Minimal RAG API Exercises

Three independent exercises using Upstage. **Each code cell runs on its own**:
you can run any exercise first or restart the kernel between exercises.

- The `openai` Python package must already be available.
- Each cell asks for your Upstage API key through a hidden prompt.
- All questions and passages are included below. No project files are required.

Run the examples, inspect the responses, and combine the pieces into your own system.


## 1. Call the LLM

Send one prompt and print the returned text. Try replacing the question.


In [ ]:
from getpass import getpass
from openai import OpenAI

client = OpenAI(
    api_key=getpass("Upstage API key: "),
    base_url="https://api.upstage.ai/v1",
)

response = client.chat.completions.create(
    model="solar-pro4-260806",
    messages=[{"role": "user", "content": "Explain RAG in two sentences."}],
    temperature=0,
    reasoning_effort="none",
    max_tokens=512,
)
print(response.choices[0].message.content)


## 2. Compare answers with and without context

Ask the same question twice with the same model settings. Only the context changes.
The passage is a teaching translation of Article 26(6), page 6, in the supplied
Ewha regulations dated **August 16, 2017**.

The passage supports **D: 90 days**. The model may also choose D without context;
compare the explanations as well as the chosen letter.


In [ ]:
from getpass import getpass
from openai import OpenAI

client = OpenAI(
    api_key=getpass("Upstage API key: "),
    base_url="https://api.upstage.ai/v1",
)

def ask(question, context=""):
    response = client.chat.completions.create(
        model="solar-pro4-260806",
        messages=[
            {"role": "system", "content": (
                "Answer the multiple-choice question using relevant context when available. "
                "If no supporting context is supplied, use your existing knowledge and say so. "
                "Return one [ANSWER]: (X) tag and a brief explanation in English."
            )},
            {"role": "user", "content": f"Question:\n{question}\n\nContext:\n{context}"},
        ],
        temperature=0,
        reasoning_effort="none",
        max_tokens=512,
    )
    return response.choices[0].message.content

question = """According to the supplied Ewha regulations (2017-08-16), within how
many days of the semester start must an enrolled student apply for leave?
(A) 30 days
(B) 45 days
(C) 60 days
(D) 90 days"""

context = """Ewha regulations (2017-08-16), Article 26(6), PDF page 6:
An enrolled student who wishes to take a leave of absence must apply
within 90 days from the beginning of the semester."""

print("WITHOUT CONTEXT\n", ask(question))
print("\nWITH CONTEXT\n", ask(question, context))


## 3. Test embeddings and retrieve a passage

Embed three inline passages and a question, then rank the passages by cosine similarity.
The passages are teaching summaries of the same supplied 2017 Ewha regulations:
Articles 26(6), 10(1), and 11(1).

Try changing the question. Then combine retrieval with the context argument of
`ask` from Exercise 2 yourself. This cell only retrieves passages.


In [ ]:
from getpass import getpass
from math import sqrt
from openai import OpenAI

client = OpenAI(
    api_key=getpass("Upstage API key: "),
    base_url="https://api.upstage.ai/v1",
)

def embed(texts, model):
    response = client.embeddings.create(model=model, input=texts, encoding_format="float")
    return [item.embedding for item in sorted(response.data, key=lambda item: item.index)]

def cosine(a, b):
    return sum(x * y for x, y in zip(a, b)) / (
        sqrt(sum(x * x for x in a)) * sqrt(sum(y * y for y in b))
    )

passages = [
    "An enrolled student must apply for leave within 90 days of the semester start.",
    "The academic year begins on March 1.",
    "The ordinary minimum number of teaching weeks per academic year is 30.",
]
question = "What is the deadline for an enrolled student to apply for leave?"

document_vectors = embed(passages, "solar-embedding-2-passage")
query_vector = embed([question], "solar-embedding-2-query")[0]
scores = [cosine(query_vector, vector) for vector in document_vectors]
ranked = sorted(zip(scores, passages), key=lambda item: item[0], reverse=True)

for rank, (score, passage) in enumerate(ranked, start=1):
    print(f"{rank}. {score:.3f}  {passage}")
